# 3. Add Knowledge

**Goal:** Give the agent Agno's native knowledge search.

In [1]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "agent.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from agno.agent import Agent
from agno.models.openai import OpenAIChat
from embedding import load_settings

RUN_LIVE = True
settings = load_settings()
model = OpenAIChat(
    id=settings["openrouter_model"],
    api_key=settings["openrouter_api_key"],
    base_url="https://openrouter.ai/api/v1",
)
print("Ready. Live model calls:", RUN_LIVE)

Ready. Live model calls: True


In [2]:
async def ask(workshop_agent, question, session_id=None):
    if not RUN_LIVE:
        return "Skipped. Set RUN_LIVE = True for a model call."
    response = await workshop_agent.arun(question, session_id=session_id)
    return response.content

In [ ]:
INSTRUCTIONS = [
    "For company financial questions, search the knowledge base first.",
    "If the knowledge base is not available, then tell the user explicitly that you cannot proceed further. Do extra message or hallunicated answer",
    "For latest figures, use the latest period in the knowledge base and name it.",
    "Do not use web search when the knowledge base answers the question.",
    "Answer PDF questions only from retrieved evidence.",
    "Cite the source filename and physical PDF page.",
    "Say when the evidence is missing.",
]

In [ ]:
from embedding import create_knowledge
knowledge = create_knowledge()
print("Knowledge ready:", knowledge.name)

Knowledge ready: Gravitas Student Knowledge


In [6]:
rag_agent = Agent(
    name="RAG Agent",
    model=model,
    knowledge=knowledge,
    search_knowledge=True,
    instructions=INSTRUCTIONS,
    markdown=True,
    telemetry=False,
)
print(await ask(rag_agent, "Share HCLTech Q1 FY27 revenue and cite the page."))

**HCLTech Q1 FY27 Revenue: $3,650 million**

- **Total Revenue:** $3,650 million, a decline of 0.5% QoQ and growth of 2.6% YoY (constant currency).
- **Services Revenue:** $3,351 million, a decline of 0.7% QoQ and growth of 3.5% YoY.
- **Advanced AI Revenue:** $171 million, marking 10.6% QoQ and 62.1% YoY growth.

**Source:** HCLTECH_Q1_FY27.pdf, page 14 (financial details provided by CFO Shiv Walia) and page 4 (commentary by CEO C. Vijayakumar).


## Check

Explain what capability this step added and which earlier limitation it fixes.